## 1. Setup

### Install required packages

In [ ]:
%pip install google-generativeai python-dotenv

### Import libraries

In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, InvalidArgument

print("Everything is working!")

### Load your API key

In [4]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


### Configure Gemini

In [ ]:
genai.configure(api_key=api_key)

MODEL_NAME = "gemini-3-flash-preview"

print("Gemini configured successfully.")

## 2. Create the Helper Function

messages → The conversation

model → Which Gemini model to use

temperature → controls how creative/random the answer is

In [6]:
def get_completion(messages, model=MODEL_NAME, temperature=0.7):
    """
    Sends messages to Gemini and returns the generated response.
    """

    # Extract system messages
    system_msgs = [
        m["content"]
        for m in messages
        if m["role"] == "system"
    ]

    system_instruction = "\n".join(system_msgs) if system_msgs else None

    # Remove system messages from chat history
    chat_msgs = [
        m for m in messages
        if m["role"] != "system"
    ]

    # Create Gemini model
    gen_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_instruction
    )

    # Convert OpenAI-style roles to Gemini roles
    history = []

    for m in chat_msgs[:-1]:
        role = "model" if m["role"] == "assistant" else "user"

        history.append({
            "role": role,
            "parts": [m["content"]]
        })

    # Last message
    last_user_msg = chat_msgs[-1]["content"]

    try:
        chat = gen_model.start_chat(history=history)

        response = chat.send_message(
            last_user_msg,
            generation_config=genai.types.GenerationConfig(
                temperature=temperature
            )
        )

        return response.text

    except ResourceExhausted:
        return "⚠️ Gemini free-tier rate limit hit. Wait a minute and try again."

    except InvalidArgument as e:
        return f"⚠️ Invalid request: {e}"

    except Exception as e:
        return f"⚠️ Unexpected error: {e}"

### Test the function

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Say hello and tell me that you are working."
    }
]

response = get_completion(messages)

print(response)

## 3. LLM Architecture Messaging

We will demonstrate:

- System message
- User message
- Assistant response

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an expert italian Chef."
    },
    {
        "role": "user",
        "content": "How do I make a delicious pizza?"
    }
]

response = get_completion(messages)

print(response)

## 4. Zero-shot Learning

Zero-shot learning means giving the model a task without providing examples.

In [ ]:
zero_shot_prompt = """
Is this sentence Happy or Sad?

"I got a new puppy today!"
"""

messages = [
    {
        "role": "user",
        "content": zero_shot_prompt
    }
]

response = get_completion(messages, temperature=0)

print(response)

## 5. Few-shot Learning

Zero-shot learning means asking the model to perform a task without giving it any examples.

In [ ]:
few_shot_prompt = """
Classify each fruit as Healthy or Unhealthy.

Fruit: Apple
Category: Healthy

Fruit: Candy
Category: Unhealthy

Fruit: Banana
Category:
"""

messages = [
    {
        "role": "user",
        "content": few_shot_prompt
    }
]

response = get_completion(messages, temperature=0)

print(response)

## 6. Chain of Thought (CoT)

Chain of Thought prompting encourages the model to solve a problem step by step.

In [ ]:
cot_prompt = """
I have 10 chocolates.
I give 3 chocolates to my friend.
Then I buy 2 more.

How many chocolates do I have?

Explain the calculation briefly.
"""

messages = [
    {
        "role": "user",
        "content": cot_prompt
    }
]

response = get_completion(messages)

print(response)

## 7. Tree of Thoughts (ToT)

Tree of Thoughts explores multiple possible solutions, compares them, and chooses the best option.

In [ ]:
tot_prompt = """
I need to travel to university.

Consider these 3 options:

1. Bus
2. Train
3. Walking

For each option, give one advantage and one disadvantage.

Then choose the best option for a student
who wants to save money and time.
"""

messages = [
    {
        "role": "user",
        "content": tot_prompt
    }
]

response = get_completion(messages)

print(response)